# Week 8 Practical: Creating and Training Neural Networks → Adam

**Dataset:** MNIST handwritten digits, loaded directly from TensorFlow/Keras with  
`tf.keras.datasets.mnist.load_data()`.

## Learning goals
By the end of this practical, you should be able to:
- load, inspect, normalize, and reshape MNIST data;
- create and train a neural network with TensorFlow/Keras;
- explain forward propagation, logits, softmax, and cross-entropy;
- distinguish sparse categorical from categorical cross-entropy;
- use numerically stable loss computation with `from_logits=True`;
- inspect predictions and evaluate a multiclass model;
- understand vectorized neural-network computation;
- compare activation functions;
- implement a simple multi-label neural network;
- explain gradient descent and the Adam optimizer;
- compare SGD and Adam and investigate Adam's learning rate.

> **Student rule:** Every numbered section contains a **TODO**. Complete the TODO before moving to the next section.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)

## 1. Load and understand MNIST

MNIST contains grayscale images of handwritten digits from 0 to 9. Keras provides a built-in loader, so no separate download link is required.

Each image has shape $(28\times28)$. The label is an integer from 0 to 9.

In [ ]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test :", X_test.shape)
print("y_test :", y_test.shape)
print("Pixel range:", X_train.min(), "to", X_train.max())

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(X_train[i], cmap="gray")
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis("off")
plt.tight_layout()
plt.show()

### TODO 1 — Explore the dataset
1. Display **15 different training images** in a $(3\times5)$ grid.
2. Print the label corresponding to every displayed image.
3. Print the number of training examples belonging to each digit (0–9).
4. In a Markdown cell, answer: **Is MNIST approximately balanced across its 10 classes?** Support your answer using the counts you obtained.

In [ ]:
# TODO 1: Write your solution here.

## 2. Preprocess the input

Neural networks usually train more effectively when input values are on a manageable numerical scale. MNIST pixels are integers from 0 to 255, so we normalize them to \([0,1]\).

For a fully connected network, each $(28\times28)$ image can also be flattened into a vector of \(784\) features.

In [ ]:
X_train_norm = X_train.astype("float32") / 255.0
X_test_norm  = X_test.astype("float32") / 255.0

X_train_flat = X_train_norm.reshape(-1, 28 * 28)
X_test_flat  = X_test_norm.reshape(-1, 28 * 28)

print(X_train_flat.shape, X_test_flat.shape)
print("Normalized range:", X_train_flat.min(), X_train_flat.max())

### TODO 2 — Verify preprocessing
1. Print the original and normalized value of pixel `[10, 10]` from the first training image.
2. Confirm that one flattened image contains **784 features**.
3. Reshape one flattened vector back to $(28\times28)$ and display it.
4. Explain why normalization can make optimization easier.

In [ ]:
# TODO 2: Write your solution here.

## 3. Creating a neural network

A dense neural network computes successive transformations

$
\mathbf a^{[l]} = g\left(W^{[l]}\mathbf a^{[l-1]}+\mathbf b^{[l]}\right).
$

For MNIST, we will use two hidden layers and 10 output units—one logit for each digit.

In [ ]:
baseline_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(25, activation="sigmoid"),
    tf.keras.layers.Dense(15, activation="sigmoid"),
    tf.keras.layers.Dense(10)   # logits
])

baseline_model.summary()

### TODO 3 — Build the network yourself
Create a new `Sequential` model named `student_model` with:
- input shape `(784,)`;
- first hidden layer: **32 neurons, ReLU**;
- second hidden layer: **16 neurons, ReLU**;
- output layer: **10 neurons with no activation**.

Then:
1. call `summary()`;
2. identify the shape of each weight matrix;
3. compute the total number of trainable parameters manually and verify it against Keras.

In [ ]:
# TODO 3: Write your solution here.

## 4. Forward propagation and logits

Before training, the network can already perform a forward pass, but its parameters are random.

The 10 output numbers are **logits**, not probabilities.

In [ ]:
sample_logits = baseline_model(X_train_flat[:5], training=False)
print(sample_logits.numpy())
print("Shape:", sample_logits.shape)

### TODO 4 — Inspect an untrained forward pass
1. Pass the first **10 examples** through `student_model`.
2. Print the output shape.
3. For the first example, identify the largest logit and its class index using `argmax`.
4. Compare this predicted class with the true label.
5. Explain why poor predictions are expected before training.

In [ ]:
# TODO 4: Write your solution here.

## 5. Softmax and multiclass probabilities

Softmax converts logits $(z_1,\ldots,z_K)$ into probabilities:

$
p_j = \frac{e^{z_j}}{\sum_{k=1}^{K}e^{z_k}}.
$

The probabilities sum to 1 because MNIST is a **multiclass** problem: one digit class is selected for each image.

In [ ]:
probs = tf.nn.softmax(sample_logits, axis=1)
print(probs.numpy())
print("Row sums:", tf.reduce_sum(probs, axis=1).numpy())

### TODO 5 — Interpret softmax
For the first five examples:
1. convert the logits from `student_model` to probabilities;
2. verify that every probability vector sums approximately to 1;
3. print the predicted class and its probability;
4. explain the difference between a **logit** and a **probability**.

In [ ]:
# TODO 5: Write your solution here.

## 6. Cross-entropy: sparse vs. one-hot labels

For a true class \(y\), multiclass cross-entropy can be written as

$
L=-\log(p_y).
$

`SparseCategoricalCrossentropy` expects integer labels such as `7`.  
`CategoricalCrossentropy` expects one-hot vectors such as `[0,0,0,0,0,0,0,1,0,0]`.

In [ ]:
example_y = y_train[:5]
example_one_hot = tf.one_hot(example_y, depth=10)

print("Integer labels:", example_y)
print("One-hot labels:\n", example_one_hot.numpy())

### TODO 6 — Compare the two cross-entropy interfaces
1. Compute loss on the same logits using:
   - `SparseCategoricalCrossentropy(from_logits=True)`;
   - `CategoricalCrossentropy(from_logits=True)` after one-hot encoding the labels.
2. Show that the two losses are approximately equal.
3. Explain why sparse categorical cross-entropy is convenient for the original MNIST labels.

In [ ]:
# TODO 6: Write your solution here.

## 7. Numerically stable loss computation

A tempting approach is:

$
\text{logits}\rightarrow\text{softmax probabilities}\rightarrow\log(p).
$

For extreme values, explicitly forming probabilities can create numerical roundoff problems. Keras can compute cross-entropy directly from logits using `from_logits=True`, which uses a numerically stable formulation.

In [ ]:
extreme_logits = tf.constant([[1000.0, -1000.0, 0.0]])
extreme_label = tf.constant([1])

stable_loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
print("Stable loss:", stable_loss(extreme_label, extreme_logits).numpy())

### TODO 7 — Investigate numerical stability
1. Apply softmax to `extreme_logits` and print the probabilities.
2. Attempt to calculate $(-\log(p_y))$ manually for the true class.
3. Compare it with TensorFlow's `SparseCategoricalCrossentropy(from_logits=True)`.
4. Explain what happens when a probability becomes numerically 0 and why $(\log(0))$ is problematic.

In [ ]:
# TODO 7: Write your solution here.

## 8. Compile and train a neural network

`compile()` specifies how the network will learn. `fit()` repeatedly performs forward propagation, loss computation, backpropagation, and parameter updates.

We first use ordinary stochastic gradient descent (SGD) so that we can later compare it with Adam.

In [ ]:
sgd_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(25, activation="relu"),
    tf.keras.layers.Dense(15, activation="relu"),
    tf.keras.layers.Dense(10)
])

sgd_model.compile(
    optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_sgd = sgd_model.fit(
    X_train_flat, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

### TODO 8 — Examine the training process
1. Plot **training loss vs. validation loss** across epochs.
2. Plot **training accuracy vs. validation accuracy**.
3. Report the final training and validation accuracy.
4. Explain what one **epoch** means and why one epoch normally contains many parameter-update steps.
5. Identify any evidence of underfitting or overfitting.

In [ ]:
# TODO 8: Write your solution here.

## 9. Evaluate and make predictions

Evaluation measures performance on data not used for parameter updates.

In [ ]:
test_loss, test_acc = sgd_model.evaluate(X_test_flat, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

test_logits = sgd_model.predict(X_test_flat[:10], verbose=0)
test_probs = tf.nn.softmax(test_logits, axis=1).numpy()
test_pred = np.argmax(test_probs, axis=1)

print("Predicted:", test_pred)
print("True     :", y_test[:10])

### TODO 9 — Analyze model errors
1. Generate predictions for the entire test set.
2. Find **10 incorrectly classified images**.
3. Display them with titles showing `True: ...` and `Predicted: ...`.
4. For at least three errors, inspect the predicted probability distribution.
5. Briefly discuss why those digits might be difficult for the model.

In [ ]:
# TODO 9: Write your solution here.

## 10. Activation functions

Common activations include:

$
\text{Sigmoid}(z)=\frac{1}{1+e^{-z}},\qquad
\text{ReLU}(z)=\max(0,z).
$

Without nonlinear activations, stacking dense layers still represents only an affine transformation.

In [ ]:
z = np.linspace(-6, 6, 300)
sigmoid = 1 / (1 + np.exp(-z))
relu = np.maximum(0, z)

plt.figure(figsize=(7,4))
plt.plot(z, sigmoid, label="Sigmoid")
plt.plot(z, relu, label="ReLU")
plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)
plt.legend()
plt.xlabel("z")
plt.ylabel("g(z)")
plt.title("Activation Functions")
plt.show()

### TODO 10 — Compare sigmoid and ReLU
1. Add **linear** and **softplus** functions to the plot.
2. Compute sigmoid values at \(z=-10,0,10\).
3. Explain sigmoid saturation and why it can lead to small gradients.
4. State which activation you would normally choose for hidden layers in this MNIST model and why.

In [ ]:
# TODO 10: Write your solution here.

## 11. Vectorization

Neural networks process many examples efficiently using matrix operations.

For a dense layer:

$
Z=XW+b,\qquad A=g(Z).
$

Here, each row of \(X\) is one example.

In [ ]:
X_small = X_train_flat[:4]
W = np.random.randn(784, 3) * 0.01
b = np.zeros((1, 3))

Z = X_small @ W + b
A = np.maximum(0, Z)

print("X:", X_small.shape)
print("W:", W.shape)
print("b:", b.shape)
print("Z:", Z.shape)
print("A:", A.shape)

### TODO 11 — Manual vectorized dense layer
1. Select 8 MNIST examples.
2. Create a random weight matrix that maps 784 inputs to 5 neurons.
3. Compute \(Z=XW+b\) using one matrix multiplication.
4. Apply ReLU.
5. Print all relevant shapes.
6. Explain why this vectorized implementation is preferable to writing nested Python loops over examples and neurons.

In [ ]:
# TODO 11: Write your solution here.

## 12. Multi-label classification

Multiclass and multi-label classification are different.

- **Multiclass:** exactly one class is correct → 10-way softmax for MNIST.
- **Multi-label:** several labels can be true simultaneously → one sigmoid output per label.

We can create a simple multi-label exercise from MNIST. For each digit, predict three properties:
1. `is_even`
2. `is_greater_than_4`
3. `has_loop` (we will define loop digits as 0, 6, 8, 9).

In [ ]:
def make_multilabel_targets(y):
    is_even = (y % 2 == 0).astype("float32")
    is_gt4 = (y > 4).astype("float32")
    has_loop = np.isin(y, [0, 6, 8, 9]).astype("float32")
    return np.stack([is_even, is_gt4, has_loop], axis=1)

Y_train_multi = make_multilabel_targets(y_train)
Y_test_multi = make_multilabel_targets(y_test)

print("Digit:", y_train[0])
print("[even, >4, loop]:", Y_train_multi[0])

### TODO 12 — Build a multi-label classifier
Create a neural network that predicts the three properties above.

Requirements:
1. use two hidden ReLU layers;
2. use **3 output neurons with sigmoid activation**;
3. compile with `binary_crossentropy`;
4. train for at least 3 epochs;
5. predict the properties for 10 test images using threshold 0.5;
6. compare predicted and true property vectors.

Finally, explain why **softmax would be inappropriate** for these three outputs.

In [ ]:
# TODO 12: Write your solution here.

## 13. From gradient descent to Adam

Gradient descent uses an update such as

$
\theta_t=\theta_{t-1}-\alpha g_t.
$

Adam maintains exponentially weighted estimates of the gradient's first and second raw moments:

$
m_t=\beta_1m_{t-1}+(1-\beta_1)g_t
$

$
v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.
$

After bias correction,

$
\hat m_t=\frac{m_t}{1-\beta_1^t},\qquad
\hat v_t=\frac{v_t}{1-\beta_2^t},
$

and the update is

$
\theta_t=\theta_{t-1}-\alpha\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.
$

Thus, $(\alpha)$ remains the **base learning rate**, while Adam scales updates adaptively using gradient history.

In [ ]:
def adam_scalar_demo(gradients, alpha=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
    m, v, theta = 0.0, 0.0, 0.0
    rows = []
    for t, g in enumerate(gradients, start=1):
        m = beta1*m + (1-beta1)*g
        v = beta2*v + (1-beta2)*(g**2)
        m_hat = m / (1-beta1**t)
        v_hat = v / (1-beta2**t)
        update = alpha * m_hat / (np.sqrt(v_hat) + eps)
        theta -= update
        rows.append((t, g, m, v, m_hat, v_hat, update, theta))
    return rows

rows = adam_scalar_demo([0.8, 0.6, 0.4, 0.2])
for row in rows:
    print(row)

### TODO 13 — Trace Adam by hand and in code
Use gradients `[1.0, 0.5, -0.25, 0.75]` with  
$(\alpha=0.001,\beta_1=0.9,\beta_2=0.999,\epsilon=10^{-8})$.

1. Modify the code to print a readable table containing $(g_t,m_t,v_t,\hat m_t,\hat v_t)$, and the parameter update.
2. Manually calculate $(m_1)$ and $(v_1)$ and verify them against the code.
3. Explain why $(m_{t-1})$ is **not simply the previous gradient**.
4. Explain what information $(v_t)$ captures.
5. State why bias correction is especially important during the first few iterations.

In [ ]:
# TODO 13: Write your solution here.

## 14. Train MNIST with Adam

Now we use the numerically stable MNIST configuration discussed in the lecture:

- 10 **linear** output units → logits;
- `SparseCategoricalCrossentropy(from_logits=True)`;
- Adam optimizer with base learning rate $(\alpha=10^{-3}=0.001)$.

In [ ]:
adam_model = tf.keras.Sequential([
    tf.keras.Input(shape=(784,)),
    tf.keras.layers.Dense(25, activation="relu"),
    tf.keras.layers.Dense(15, activation="relu"),
    tf.keras.layers.Dense(10)
])

adam_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

history_adam = adam_model.fit(
    X_train_flat, y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1,
    verbose=1
)

adam_test_loss, adam_test_acc = adam_model.evaluate(
    X_test_flat, y_test, verbose=0
)
print(f"Adam test accuracy: {adam_test_acc:.4f}")

### TODO 14 — Compare SGD and Adam
Using the SGD model from Section 8 and the Adam model above:

1. plot their training losses on the same graph;
2. plot their validation accuracies on the same graph;
3. report both test accuracies;
4. identify which optimizer converged faster in your experiment;
5. explain why Adam can have different effective step sizes for different parameters even though a single base learning rate was supplied.

In [ ]:
# TODO 14: Write your solution here.

## 15. Adam learning-rate experiment

Adam adapts parameter updates, but the base learning rate \(\alpha\) still controls their overall scale.

### TODO 15 — Investigate the base learning rate
Train the **same architecture from the same random initialization strategy** using Adam with:

$
\alpha\in\{10^{-4},10^{-3},10^{-2}\}.
$

For each learning rate:
1. train for 5 epochs using the same batch size;
2. record final training loss, validation loss, validation accuracy, and test accuracy;
3. plot validation accuracy for all three runs on one graph;
4. identify which learning rate worked best;
5. explain why Adam still needs a base learning rate even though it adaptively scales updates.

**Important:** Reset the TensorFlow random seed before constructing each model so that your comparison is as fair as possible.

In [ ]:
# TODO 15: Write your experiment here.
learning_rates = [1e-4, 1e-3, 1e-2]

## 16. Final synthesis challenge

This final task combines model design, stable loss computation, training, optimization, and evaluation.

### TODO 16 — Build your best Week 8 MNIST model
Design and train your own fully connected MNIST classifier subject to these constraints:

- use **2–4 hidden layers**;
- use a nonlinear activation in every hidden layer;
- output **10 logits**;
- use `SparseCategoricalCrossentropy(from_logits=True)`;
- use Adam;
- train for no more than **10 epochs**.

Then:
1. justify your architecture and learning rate;
2. report training, validation, and test performance;
3. plot the learning curves;
4. show 10 test predictions, including at least 5 errors if your model makes that many;
5. convert logits to probabilities only when you need probabilities for interpretation;
6. write a short conclusion explaining what you learned about architecture, activation functions, stable loss computation, and Adam.

In [ ]:
# TODO 16: Build your final model here.

---
## Practical checklist

Before submission, confirm that you have:
- completed **TODOs 1–16**;
- executed the notebook from top to bottom without unresolved errors;
- included all requested plots and numerical results;
- answered all conceptual questions in Markdown cells;
- clearly distinguished **logits**, **probabilities**, and **class predictions**;
- clearly distinguished **multiclass** from **multi-label** classification;
- explained the roles of $(m_t)$, $(v_t)$, and the base learning rate $(\alpha)$ in Adam.